In [43]:
import os
import requests


In [21]:
model = "openai/gpt-3.5-turbo"
url = "https://openrouter.ai/api/v1/chat/completions"

In [22]:
def call_llm(system_prompt , user_prompt, temperature=0.0, max_token = 512):
    api_key = os.environ.get("API_KEY")
    if api_key is None:
        print (f"API key not found")
        return None

    payload = {
        "model":model,
        "messages" : [
            {
                "role" : "system",
                "content" : system_prompt
            },
            {
                "role":"user",
                "content":user_prompt
            }
        ],
        "max_tokens" : max_token,
        "temperature": temperature
    }

    headers = {
        "Authorization":f"Bearer {api_key}",
        "Content-Type":"application/json"
    }
    
    try:
        response = requests.post(url, headers = headers, json=payload)
    except requests.exceptions.ConnectionError as err:
        print(f"Connection Error: {err}")
        return None
    except requests.exceptions.Timeout as err:
        print(f"Request Timed Out: {err}")
        return None
    except requests.exceptions.RequestException as err:
        print(f"Request Failed: {err}")
        return None

    if response.status_code != 200:
        print(f"Error : {response.status_code}")
        print(response.text)
        return None
    try:
        return response.json()["choices"][0]["message"]["content"]
    except (KeyError, IndexError, ValueError):
        print("Unexpected API response:")
        print(response.json())
    return None

In [23]:
system_prompt = "You are a helpful assistant."
user_prompt = "Reply with only the word: hello"

response = call_llm(system_prompt, user_prompt)

print(response)

Hello


In [44]:
#Define SYSTEM PROMPT

system_prompt = """
    You are a healthcare lifestyle recommendation assistant.

    your task is to generate personalized lifestyle suggestions based on patient's health record.
    Constraints:
        Do not diagnose disease. Do not prescribe medicines.
        if bmi > 30, suggest some weight management techniques
        if hypertension =1, suggest BP monitoring and provide some excercies/yoga to reduce the BP
        if glucose_level >140, suggest to reduce the sugar intake
        if smonking_status indicates smoking, then suggeet to quite smoking or actions can be taken to reduce it.
        Always include medical disclaimer. Return only valid json data.

        Example:

        input :
        { "age" : 68,
        "hypertension":1,
        "heart_disease":0,
        "avg_glucose_level":180,
        "bmi":31,
        "smoking_status":"formerly smoked"
        }
        
        output:
        {
        "diet" : ["reduce sugar intake","reduce salt intake"],
        "exercise":["walk daily for 30 mins"];
        "healthy_habits":["monitor bp","maintain healthy weight", "proper food diet"],
        "warning":""
        }

        Do not rename any key in the output, do not include additional fields
        """


user_prompt = """
    Patient Profile

    Age: 65
    Hypertension: 1
    Heart Disease: 0
    Average Glucose Level: 180
    BMI: 31
    Smoking Status: formerly smoked

    Generate personalized lifestyle recommendations in JSON only.
"""


response = call_llm(system_prompt, user_prompt, temperature=0)
print(f"Temperature : 0 ")
print(f"Response : {response}")

print()
response2 = call_llm(system_prompt, user_prompt, temperature=0.7)
print(f"Temperature : 0.7")
print(f"Response : {response2}")

Temperature : 0 
Response : {
    "diet": ["reduce sugar intake", "reduce salt intake"],
    "exercise": ["walk daily for 30 mins"],
    "healthy_habits": ["monitor bp", "maintain healthy weight", "proper food diet"],
    "warning": "It is recommended to consult with a healthcare professional for personalized advice. This information is for general guidance only."
}

Temperature : 0.7
Response : {
    "diet": ["reduce sugar intake", "reduce salt intake"],
    "exercise": ["walk daily for 30 mins"],
    "healthy_habits": ["monitor bp", "maintain healthy weight", "proper food diet"],
    "warning": "Please note that these are general lifestyle recommendations. Consult with a healthcare professional for personalized advice."
}


In [46]:
# 3 differnet prompts

print(f"Prompt :1")
patient1 = """
Age:45
Hypertension:0
Heart Disease:0
Average Glucose Level:95
BMI:23
Smoking Status:never smoked
"""
response = call_llm(system_prompt, patient1, temperature=0)
print(f"Temperature : 0 ")
print(f"Response : {response}")
print()

response2 = call_llm(system_prompt, patient1, temperature=0.7)
print(f"Temperature : 0.7 ")

print(f"Response : {response2}")
print("="*20)

patient2 = """
Age:65
Hypertension:1
Heart Disease:0
Average Glucose Level:180
BMI:31
Smoking Status:formerly smoked
"""
print(f"Prompt :2")
response = call_llm(system_prompt, patient2, temperature=0)
print(f"Temperature : 0 ")
print(f"Response : {response}")
print()

response2 = call_llm(system_prompt, patient2, temperature=0.7)
print(f"Temperature : 0.7")
print(f"Response : {response2}")
print("="*20)
print()
patient3 = """
Age:72
Hypertension:1
Heart Disease:1
Average Glucose Level:210
BMI:34
Smoking Status:smokes
"""

print(f"Prompt : 3")
response = call_llm(system_prompt, patient3, temperature=0)
print(f"Temperature : 0 ")

print(f"Response : {response}")
print()

response2 = call_llm(system_prompt, patient3, temperature=0.7)
print(f"Temperature : 0.7 ")

print(f"Response : {response2}")

Prompt :1
Temperature : 0 
Response : {
    "diet": [],
    "exercise": [],
    "healthy_habits": ["maintain healthy weight"],
    "warning": "Always consult with a healthcare professional for personalized medical advice."
}

Temperature : 0.7 
Response : {
    "diet": [],
    "exercise": [],
    "healthy_habits": ["regular health check-ups", "maintain a healthy weight"],
    "warning": ""
}
Prompt :2
Temperature : 0 
Response : {
    "diet": ["reduce sugar intake", "reduce salt intake"],
    "exercise": ["walk daily for 30 mins"],
    "healthy_habits": ["monitor bp", "maintain healthy weight", "proper food diet"],
    "warning": "It is recommended to consult with a healthcare professional for personalized advice. This information is not a substitute for medical consultation."
}

Temperature : 0.7
Response : {
    "diet": ["reduce sugar intake", "reduce salt intake"],
    "exercise": ["practice yoga daily for 30 mins"],
    "healthy_habits": ["monitor blood pressure regularly", "mainta

In [26]:
from jsonschema import validate

recommendation_schema = {
    "type": "object",
    "properties": {
        "diet": {
            "type": "array",
            "items": {"type": "string"}
        },
        "exercise": {
            "type": "array",
            "items": {"type": "string"}
        },
        "healthy_habits": {
            "type": "array",
            "items": {"type": "string"}
        },
        "warning": {
            "type": "string"
        }
    },
    "required": [
        "diet",
        "exercise",
        "healthy_habits",
        "warning"
    ],
    "additionalProperties": False
}

In [37]:
fallback = {
    "diet": None,
    "exercise": None,
    "healthy_habits": None,
    "warning": None
}


#Prepare three dataset records
records = [
    {
        "age": 45,
        "hypertension": 0,
        "heart_disease": 0,
        "avg_glucose_level": 95,
        "bmi": 23,
        "smoking_status": "never smoked"
    },
    {
        "age": 65,
        "hypertension": 1,
        "heart_disease": 0,
        "avg_glucose_level": 180,
        "bmi": 31,
        "smoking_status": "formerly smoked"
    },
    {
        "age": 72,
        "hypertension": 1,
        "heart_disease": 1,
        "avg_glucose_level": 210,
        "bmi": 34,
        "smoking_status": "smokes"
    },
    {
        "age": 29,
        "hypertension": 1,
        "heart_disease": 0,
        "avg_glucose_level": 100,
        "bmi": 24,
        "smoking_status": "never smoked",
        "email" : "Idnhu@gmail.com"
    },
]

In [41]:

#Call LLM and validate


import json
from jsonschema import validate, ValidationError

for i, record in enumerate(records, start=1):

    user_prompt = json.dumps(record, indent=2)

    raw_response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print(f"\nRecord {i}")
    print("Input:")
    print(user_prompt)
    print("\nRaw LLM Response:")
    print(raw_response)

    try:
        response_json = json.loads(raw_response.strip())

        validate(
            instance=response_json,
            schema=recommendation_schema
        )

        print("Validation: SUCCESS")

    except json.JSONDecodeError as err:

        print("JSON Parsing Error:", err)

        response_json = fallback

        print("Validation: FAILED")

    except ValidationError as err:

        print("Schema Validation Error:", err.message)

        response_json = fallback

        print("Validation: FAILED")

    print("\nFinal Output")
    print(response_json)


Record 1
Input:
{
  "age": 45,
  "hypertension": 0,
  "heart_disease": 0,
  "avg_glucose_level": 95,
  "bmi": 23,
  "smoking_status": "never smoked"
}

Raw LLM Response:
{
    "diet": [],
    "exercise": [],
    "healthy_habits": ["regular health check-ups"],
    "warning": ""
}
Validation: SUCCESS

Final Output
{'diet': [], 'exercise': [], 'healthy_habits': ['regular health check-ups'], 'warning': ''}

Record 2
Input:
{
  "age": 65,
  "hypertension": 1,
  "heart_disease": 0,
  "avg_glucose_level": 180,
  "bmi": 31,
  "smoking_status": "formerly smoked"
}

Raw LLM Response:
{
    "diet": ["reduce sugar intake", "reduce salt intake"],
    "exercise": ["practice yoga for stress management", "walk daily for 30 mins"],
    "healthy_habits": ["monitor bp", "maintain healthy weight", "proper food diet"],
    "warning": "These lifestyle suggestions are based on the provided health record. Please consult a healthcare professional for personalized advice."
}
Validation: SUCCESS

Final Output
{

In [36]:
#PII detection function

import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

#wrapper method for call_llm

def safe_call_llm(system_prompt,
                  user_prompt,
                  temperature=0.0,
                  max_tokens=512):

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None

    return call_llm(
        system_prompt,
        user_prompt,
        temperature,
        max_tokens
    )

#guardrail : Test1 should be blocked
print(f"Test case 1 : Should be Blocked")
user_prompt = """
Patient Name: John

Email: john.doe@gmail.com

Age: 65
BMI: 31
"""

response = safe_call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

print(response)
print()
print(f"Test case-II should be passed")
#guardrail : TEst2 should be passed
user_prompt = """
Age: 65

Hypertension: 1

Heart Disease: 0

Average Glucose Level: 180

BMI: 31

Smoking Status: formerly smoked
"""

response = safe_call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

print(response)

Test case 1 : Should be Blocked
Input blocked: PII detected.
None

Test case-II should be passed
{
    "diet": ["reduce sugar intake", "reduce salt intake"],
    "exercise": ["engage in light cardio exercises like swimming or cycling"],
    "healthy_habits": ["monitor blood pressure regularly", "maintain a healthy weight", "follow a balanced diet"],
    "warning": "These lifestyle suggestions are not a substitute for professional medical advice. Please consult a healthcare provider for personalized recommendations."
}


In [42]:

#Call LLM and validate


import json
from jsonschema import validate, ValidationError

for i, record in enumerate(records, start=1):

    user_prompt = json.dumps(record, indent=2)

    raw_response = safe_call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print(f"\nRecord {i}")
    print("Input:")
    print(user_prompt)

    if raw_response is None:
        print(f"\nGuardrail Result : BLOCKED , PII detected")
        print(f"JSON Validation :Not Performed")
        continue

    print("\nGuardrail Result: PASSED")
    print("\nRaw LLM Response:")
    print(raw_response)

    try:
        response_json = json.loads(raw_response.strip())

        validate(
            instance=response_json,
            schema=recommendation_schema
        )

        print("Validation: SUCCESS")

    except json.JSONDecodeError as err:

        print("JSON Parsing Error:", err)

        response_json = fallback

        print("Validation: FAILED")

    except ValidationError as err:

        print("Schema Validation Error:", err.message)

        response_json = fallback

        print("Validation: FAILED")

    print("\nFinal Output")
    print(response_json)


Record 1
Input:
{
  "age": 45,
  "hypertension": 0,
  "heart_disease": 0,
  "avg_glucose_level": 95,
  "bmi": 23,
  "smoking_status": "never smoked"
}

Guardrail Result: PASSED

Raw LLM Response:
{
    "diet": [],
    "exercise": [],
    "healthy_habits": ["regular health check-ups"],
    "warning": ""
}
Validation: SUCCESS

Final Output
{'diet': [], 'exercise': [], 'healthy_habits': ['regular health check-ups'], 'warning': ''}

Record 2
Input:
{
  "age": 65,
  "hypertension": 1,
  "heart_disease": 0,
  "avg_glucose_level": 180,
  "bmi": 31,
  "smoking_status": "formerly smoked"
}

Guardrail Result: PASSED

Raw LLM Response:
{
    "diet": ["reduce sugar intake", "reduce salt intake"],
    "exercise": ["practice yoga for stress management", "walk daily for 30 mins"],
    "healthy_habits": ["monitor bp", "maintain healthy weight", "proper food diet"],
    "warning": "These lifestyle suggestions are based on the provided information. Please consult a healthcare professional for personali